In [2]:
# [CELL 1]
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
n_samples = 1500

# 1. Fitur Input: EC25, Suhu, Delta_EC
ec25_vals = np.random.uniform(4.0, 7.2, n_samples)
temp_vals = np.random.uniform(22.0, 36.0, n_samples)
delta_vals = np.random.uniform(0.01, 0.65, n_samples)

X_data = np.column_stack([ec25_vals, temp_vals, delta_vals])

# 2. Target Output: Mutu (Klasifikasi) dan Sisa Waktu (Regresi)
y_mutu = np.zeros(n_samples, dtype=np.int32)
y_waktu = np.zeros(n_samples, dtype=np.float32)

for i in range(n_samples):
    if ec25_vals[i] < 5.4 and delta_vals[i] < 0.15:
        y_mutu[i] = 0
        y_waktu[i] = np.random.uniform(180.0, 300.0)
    elif ec25_vals[i] < 6.2:
        y_mutu[i] = 1
        y_waktu[i] = np.random.uniform(30.0, 160.0)
    else:
        y_mutu[i] = 2
        y_waktu[i] = 0.0

print(f"Dataset berhasil dibuat: {X_data.shape[0]} baris sampel.")
print("Contoh data baris pertama:", X_data[0], "Label:", y_mutu[0], "Waktu:", y_waktu[0])

Dataset berhasil dibuat: 1500 baris sampel.
Contoh data baris pertama: [ 5.19852838 29.26714499  0.44052992] Label: 1 Waktu: 107.908905


In [3]:
# [CELL 2]
# Batas minimum dan maksimum fitur untuk normalisasi manual di mikrokontroler
NORM_MIN = np.array([4.0, 20.0, 0.0], dtype=np.float32)
NORM_MAX = np.array([7.5, 40.0, 0.7], dtype=np.float32)

X_scaled = (X_data - NORM_MIN) / (NORM_MAX - NORM_MIN)
X_scaled = np.clip(X_scaled, 0.0, 1.0).astype(np.float32)

# Pembagian data latih (80%) dan data uji (20%)
split_idx = int(0.8 * n_samples)

X_train, X_val = X_scaled[:split_idx], X_scaled[split_idx:]
y_mutu_train, y_mutu_val = y_mutu[:split_idx], y_mutu[split_idx:]
y_waktu_train, y_waktu_val = y_waktu[:split_idx], y_waktu[split_idx:]

print(f"Data latih: {X_train.shape[0]} | Data validasi: {X_val.shape[0]}")

Data latih: 1200 | Data validasi: 300


In [4]:
# [CELL 3]
# Input layer (3 fitur)
input_layer = keras.Input(shape=(3,), name="sensor_input")

# Hidden layers (Shallow Network)
dense_1 = layers.Dense(16, activation="relu", name="dense_1")(input_layer)
dense_2 = layers.Dense(8, activation="relu", name="dense_2")(dense_1)

# Cabang 1: Klasifikasi Mutu (3 neuron, probabilitas Softmax)
out_mutu = layers.Dense(3, activation="softmax", name="output_mutu")(dense_2)

# Cabang 2: Regresi Sisa Waktu (1 neuron, nilai kontinu linear)
out_waktu = layers.Dense(1, activation="linear", name="output_waktu")(dense_2)

model = keras.Model(inputs=input_layer, outputs=[out_mutu, out_waktu], name="TinyML_Milk_Model")
model.summary()

E0000 00:00:1789318009.127246  411647 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "TinyML_Milk_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sensor_input        │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │         64 │ sensor_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 8)         │        136 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_mutu (Dense) │ (None, 3)         │         27 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_waktu        │ (None, 1)         │          9 │ dense_2[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 236 (944.00 B)

 Trainable params: 236 (944.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
# [CELL 4]
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005),
    loss={
        "output_mutu": "sparse_categorical_crossentropy",
        "output_waktu": "mean_squared_error"
    },
    loss_weights={
        "output_mutu": 1.0,
        "output_waktu": 0.01  # Mengimbangi skala angka puluhan/ratusan pada regresi
    },
    metrics={
        "output_mutu": "accuracy",
        "output_waktu": "mae"
    }
)

history = model.fit(
    X_train,
    {"output_mutu": y_mutu_train, "output_waktu": y_waktu_train},
    validation_data=(X_val, {"output_mutu": y_mutu_val, "output_waktu": y_waktu_val}),
    epochs=50,
    batch_size=32,
    verbose=1
)

Epoch 1/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 118.6743 - output_mutu_accuracy: 0.6075 - output_mutu_loss: 0.9633 - output_waktu_loss: 11719.9482 - output_waktu_mae: 79.1039 - val_loss: 115.5140 - val_output_mutu_accuracy: 0.5800 - val_output_mutu_loss: 0.9687 - val_output_waktu_loss: 11356.2480 - val_output_waktu_mae: 79.6325
Epoch 2/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 106.1510 - output_mutu_accuracy: 0.6058 - output_mutu_loss: 1.0393 - output_waktu_loss: 10566.0635 - output_waktu_mae: 76.3966 - val_loss: 94.1611 - val_output_mutu_accuracy: 0.6300 - val_output_mutu_loss: 0.9857 - val_output_waktu_loss: 9173.9023 - val_output_waktu_mae: 74.2970
Epoch 3/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 80.4602 - output_mutu_accuracy: 0.5858 - output_mutu_loss: 0.8801 - output_waktu_loss: 7896.9883 - output_waktu_mae: 69.0222 - val_loss: 70.8969 - val_output_mutu_accuracy: 0.7200 - val_output_mutu_loss: 0.8081 - val_output_waktu_loss: 6772.6963 - val_output_wakt

In [6]:
# [CELL 5]
def representative_dataset_gen():
    for i in range(100):
        # Menyediakan potongan data riil berskala untuk kalibrasi rentang kuantisasi
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen

# Membatasi operasi agar terkuantisasi penuh ke INT8
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.float32   # Mempertahankan input float agar praktis di firmware
converter.inference_output_type = tf.float32  # Mempertahankan output float

tflite_quant_model = converter.convert()

# Simpan model terkompresi
tflite_filename = "tinyml_milk_quant.tflite"
with open(tflite_filename, "wb") as f:
    f.write(tflite_quant_model)

print(f"Model berhasil dikonversi ke format TFLite INT8.")
print(f"Ukuran akhir file biner: {len(tflite_quant_model)} byte ({len(tflite_quant_model)/1024:.2f} KB)")

INFO:tensorflow:Assets written to: /tmp/tmpc0iv6xxv/assets


INFO:tensorflow:Assets written to: /tmp/tmpc0iv6xxv/assets


Saved artifact at '/tmp/tmpc0iv6xxv'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 3), dtype=tf.float32, name='sensor_input')
Output Type:
  List[TensorSpec(shape=(None, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)]
Captures:
  140410104181536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140410104182064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140410104179952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140410104176784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140410104180128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140410100762576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140410104182944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140410104177664: TensorSpec(shape=(), dtype=tf.resource, name=None)
Model berhasil dikonversi ke format TFLite INT8.
Ukuran akhir file biner: 4616 byte (4.51 KB)


/home/ridho/ngoding/Maestro Fest/pelatihan/.venv/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1789318060.011236  411647 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1789318060.011261  411647 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1789318060.011731  411647 reader.cc:83] Reading SavedModel from: /tmp/tmpc0iv6xxv
I0000 00:00:1789318060.012647  411647 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1789318060.012660  411647 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpc0iv6xxv
I0000 00:00:1789318060.021064  411647 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1789318060.022451  411647 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1789318060.074014  411647 loader.cc:220] Running initi

In [7]:
# [CELL 6]
def generate_c_header(tflite_data, output_path="model_data.h"):
    size = len(tflite_data)
    with open(output_path, "w") as f:
        f.write("#ifndef MODEL_DATA_H\n")
        f.write("#define MODEL_DATA_H\n\n")
        f.write("#include <stdint.h>\n\n")
        f.write(f"const unsigned int g_model_len = {size};\n")
        f.write("alignas(16) const unsigned char g_model[] = {\n  ")
        
        for idx, byte in enumerate(tflite_data):
            f.write(f"0x{byte:02x}, ")
            if (idx + 1) % 12 == 0:
                f.write("\n  ")
                
        f.write("\n};\n\n")
        f.write("#endif // MODEL_DATA_H\n")
        
    print(f"Header file C++ berhasil digenerate pada: {output_path}")

generate_c_header(tflite_quant_model)

Header file C++ berhasil digenerate pada: model_data.h


In [8]:
# [CELL 7]
# Inisialisasi TFLite Interpreter
interpreter = tf.lite.Interpreter(model_path="tinyml_milk_quant.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Simulasi data uji: Susu Mulai Asam (EC25 = 5.8 mS/cm, Suhu = 29.5 °C, Delta_EC = 0.28)
sample_test = np.array([[5.8, 29.5, 0.28]], dtype=np.float32)
sample_test_scaled = (sample_test - NORM_MIN) / (NORM_MAX - NORM_MIN)
sample_test_scaled = np.clip(sample_test_scaled, 0.0, 1.0).astype(np.float32)

# Jalankan inferensi
interpreter.set_tensor(input_details[0]['index'], sample_test_scaled)
interpreter.invoke()

# Ekstrak kedua output
pred_mutu = interpreter.get_tensor(output_details[0]['index'])
pred_waktu = interpreter.get_tensor(output_details[1]['index'])

label_map = {0: "Grade A", 1: "Grade B (Early Warning)", 2: "Grade C (Rejected)"}
pred_class = np.argmax(pred_mutu[0])

print("=== HASIL UJI INFERENSI MODEL TFLITE INT8 ===")
print(f"Prediksi Mutu        : {label_map[pred_class]} (Probabilitas: {pred_mutu[0][pred_class]*100:.1f}%)")
print(f"Estimasi Sisa Waktu  : {pred_waktu[0][0]:.1f} Menit")

=== HASIL UJI INFERENSI MODEL TFLITE INT8 ===
Prediksi Mutu        : Grade A (Probabilitas: 8447.1%)
Estimasi Sisa Waktu  : 0.0 Menit


/home/ridho/ngoding/Maestro Fest/pelatihan/.venv/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
